In [1]:
import mpmath as mp
mp.mp.dps = 40

# Exact Schwarzschild periastron advance via the orbit integral.
# Units G=c=1, central mass M. Semi-latus rectum p, eccentricity e.
# u = 1/r ; turning points u1=(1+e)/p (perihelion), u2=(1-e)/p (aphelion)
# orbit eqn: (du/dphi)^2 = 2M u^3 - u^2 + (2M/L^2)u + (E^2-1)/L^2
#   = 2M (u-u1)(u-u2)(u-u3) with u3 = 1/(2M) - (u1+u2)
# Total angle swept perihelion->perihelion:
# dphi = du / sqrt( 2M (u-u2)(u1-u)(u3-u) )  for u in (u2,u1)
# advance = 2 * integral - 2*pi

def advance(M, p, e):
    u1 = (1+e)/p
    u2 = (1-e)/p
    u3 = 1/(2*M) - (u1+u2)
    # substitution u = u2 + (u1-u2)*sin^2(psi), psi in [0,pi/2]
    # du = (u1-u2)*2 sin psi cos psi dpsi
    # (u-u2) = (u1-u2) sin^2, (u1-u) = (u1-u2) cos^2
    def integrand(psi):
        s = mp.sin(psi); cc = mp.cos(psi)
        u = u2 + (u1-u2)*s**2
        du_factor = (u1-u2)*2*s*cc
        denom = mp.sqrt(2*M*(u1-u2)**2*s**2*cc**2*(u3-u))
        return du_factor/denom
    half = mp.quad(integrand, [0, mp.pi/2])
    return 2*half - 2*mp.pi

# Fit advance(M) = A*(M/p) + B*(M/p)^2 + C*(M/p)^3 for fixed e, small M/p
e = mp.mpf('0.2056')   # Mercury-like
p = mp.mpf(1)          # set semi-latus rectum = 1, vary M
xs = [mp.mpf('1e-3'), mp.mpf('2e-3'), mp.mpf('3e-3'), mp.mpf('4e-3'), mp.mpf('5e-3')]
# advance as function of x = M/p (here p=1 so x=M)
data = [(x, advance(x, p, e)) for x in xs]

# Solve for A,B,C via least squares-ish (5 pts, 3 unknowns) using polyfit on x
X = mp.matrix([[x, x**2, x**3] for x,_ in data])
Y = mp.matrix([adv for _,adv in data])
# normal equations
XT = X.T
coef = mp.lu_solve(XT*X, XT*Y)
A,B,C = coef[0],coef[1],coef[2]
print("Exact Schwarzschild periastron advance, series in (M/p):")
print("  A (coeff of M/p)   =", mp.nstr(A,12), "   expected 6*pi =", mp.nstr(6*mp.pi,12))
print("  B (coeff of (M/p)^2)=", mp.nstr(B,12))
print("  sign of B (GR 2nd order):", "POSITIVE" if B>0 else "NEGATIVE")
print()

# RG precession: Dphi_RG = 6 pi M/p - 4 pi M^2 (1-e^2)/p^2
A_RG = 6*mp.pi
B_RG = -4*mp.pi*(1-e**2)   # coeff of (M/p)^2  (with p=1)
print("RG precession, series in (M/p):")
print("  A_RG =", mp.nstr(A_RG,12))
print("  B_RG =", mp.nstr(B_RG,12), "  sign:", "POSITIVE" if B_RG>0 else "NEGATIVE")
print()
print("=> First-order coefficients agree:", mp.nstr(A,8), "vs", mp.nstr(A_RG,8))
print("=> Second-order coefficients: GR =", mp.nstr(B,8), " RG =", mp.nstr(B_RG,8))
print("=> They have OPPOSITE SIGN." if B*B_RG < 0 else "=> Same sign.")

Exact Schwarzschild periastron advance, series in (M/p):
  A (coeff of M/p)   = 18.8496190263    expected 6*pi = 18.8495559215
  B (coeff of (M/p)^2)= 84.951043971
  sign of B (GR 2nd order): POSITIVE

RG precession, series in (M/p):
  A_RG = 18.8495559215
  B_RG = -12.0351730382   sign: NEGATIVE

=> First-order coefficients agree: 18.849619 vs 18.849556
=> Second-order coefficients: GR = 84.951044  RG = -12.035173
=> They have OPPOSITE SIGN.
